<a href="https://colab.research.google.com/github/LowRands/Machine-Learning-with-python/blob/main/Campus_Dataset_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Campus Dataset Project
Ethan Lourens

## Data Preparation
This section loads the campus dataset and prepares it for analysis


In [1]:
import pandas as pd

### Load the dataset

In [2]:
df = pd.read_excel(
    '/content/drive/MyDrive/Colab Notebooks/CampusWalkDataset_TemplateIDs.xlsx',
    sheet_name=0, #reading the first sheet from the excel spreadsheet
    skiprows = 4) #skipping the first 4 rows because they are blank

In [3]:
#display(df)

In [4]:
#df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CampusWalkDataset.csv',
   # skiprows=4,
#)

In [5]:
#pd.set_option('display.max_columns', None)
##pd.set_option('display.max_rows', None)
#pd.set_option('display.max_colwidth', None)

#display(df)

In [6]:
print(df.isnull().sum()) # Checking each catergory for no entries, adding them up and displaying the amount per category

recorder_id                                0
date                                      64
location                                  64
weather                                   64
temperature_c                             64
sound level                               64
foot_traffic                              64
primary_sound_type_urban                  64
other_sound_type_urban                    67
primary_sound_type_nature                 64
other_sound_type_nature                   66
insects_count                             64
birds_count                               64
trees_presence                            69
native_or_wild_plant_presence             76
ornnamental_or_planted_plant_presence    102
moss_or_lichen_presence                  100
wellbeing_rating                          64
ease_rating                               64
stress_rating                             64
nature_connection_experience_rating       64
most_notable_nature_element               79
other_note

In [7]:
df = df.dropna(
    subset=df.columns.difference(['recorder_id']),
    how='all'
) # deletes the rows that has no entries other than recorder id

In [8]:
duplicates = df.duplicated().sum()
print("Exact duplicate rows:", duplicates) # deletes all rows that have the exact same rows

Exact duplicate rows: 0


In [9]:
print("Rows:", len(df))
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Rows: 135
Missing values: 179
Duplicate rows: 0


# Cleaning up the Dates
This code converts the entries in the date column from their original format (like strings or mixed types) into standardized pandas datetime objects (YYYY-MM-DD).

The errors="coerce" parameter ensures that if there are any invalid dates, corrupt texts, or empty values, pandas will gracefully convert them to NaT (Not a Time/null values) instead of crashing with an error.

In [10]:
df.date = pd.to_datetime(
    df.date,
    errors="coerce"
)
display(df.date)

,date
0,2026-09-22
1,2026-09-22
2,2026-09-22
3,2026-09-22
4,2026-09-22
...,...
133,2026-09-23
134,2026-10-07
135,2026-10-07
136,2026-10-07


# Cleaning up the Weather column
- Changing the weather column to all have the same formatting
- Having each word start with a capital, so (**partly cloudy**) changes to (**Partly Cloudy**) and (**sunny**) changes to (**Sunny**)

In [11]:
# "str" converts the entire column to strings and "title" converts every string to *Title Case*, the first letter of every word is capitalised
df.weather = df.weather.str.title()
display(df.weather)

,weather
0,Partly Cloudy
1,Sunny
2,Partly Cloudy
3,Partly Cloudy
4,Sunny
...,...
133,Windy
134,Windy
135,Windy
136,Partly Cloudy


In [12]:
# looks for weather that is outside the valid weather array.
valid_weather = [
    "Sunny",
    "Partly Cloudy",
    "Cloudy",
    "Windy",
    "Raining"
]

invalid = df[~df["weather"].isin(valid_weather)]
print(invalid[["weather"]])

Empty DataFrame
Columns: [weather]
Index: []


In [13]:
weather_mapping = {
    'Sunny': 0,
    'Partly Cloudy': 1,
    'Cloudy': 2,
    'Windy': 3,
    'Raining': 4
}

df['Weather_mapping_encoded'] = df['weather'].map(weather_mapping)
print("Mapping Verification")
print(df[['weather', 'Weather_mapping_encoded']].drop_duplicates().reset_index(drop=True))

Mapping Verification
         weather  Weather_mapping_encoded
0  Partly Cloudy                        1
1          Sunny                        0
2          Windy                        3
3         Cloudy                        2


# Cleaning up the Temperature Column
- I am changing the entire temperature column to all have the same formatting
- All Temperature is in Celcuis  $(^\circ\mathrm{C})$

In [14]:
# Clean any duplicate 'C', '°', or '°C' characters from the temperature column, then format uniformly
df['temperature_c'] = (
    df['temperature_c']
    .astype(str)
    .str.replace('°C', '', regex=False)
    .str.replace('°', '', regex=False)
    .str.replace('C', '', regex=False)
    .str.strip()
)
# Re-apply uniform format for non-null/non-NaN entries
df['temperature_c'] = df['temperature_c'].apply(
    lambda x: f"{x}°C" if pd.notnull(x) and x != 'nan' and x != '' else None
)
display(df.temperature_c)

,temperature_c
0,16°C
1,16°C
2,16°C
3,16°C
4,16°C
...,...
133,16°C
134,10°C
135,10°C
136,15°C


# Cleaning the Bird and Insect count columns
- Makes the bird/insect numbers into integers instead of floats
- Checks if any of the bird/insect inputted data is invalid/negative numbers


In [15]:
df['birds_count'] = df['birds_count'].astype('Int64')
df['insects_count'] = df['insects_count'].astype('Int64')

print(df["birds_count"].describe())
print(df["insects_count"].describe())

count       135.0
mean     1.992593
std      2.789897
min           0.0
25%           0.0
50%           1.0
75%           3.0
max          20.0
Name: birds_count, dtype: Float64
count       135.0
mean     2.733333
std      5.106624
min           0.0
25%           0.0
50%           1.0
75%           3.0
max          31.0
Name: insects_count, dtype: Float64


In [16]:
# checks if bird and insect values are less than 0/invalid
invalid_birds = df[df["birds_count"] < 0]
print(invalid_birds)
invalid_insects = df[df["insects_count"] < 0]
print(invalid_insects)

Empty DataFrame
Columns: [recorder_id, date, location, weather, temperature_c, sound level, foot_traffic, primary_sound_type_urban, other_sound_type_urban, primary_sound_type_nature, other_sound_type_nature, insects_count, birds_count, trees_presence, native_or_wild_plant_presence, ornnamental_or_planted_plant_presence, moss_or_lichen_presence, wellbeing_rating, ease_rating, stress_rating, nature_connection_experience_rating, most_notable_nature_element, other_notes, Weather_mapping_encoded]
Index: []

[0 rows x 24 columns]
Empty DataFrame
Columns: [recorder_id, date, location, weather, temperature_c, sound level, foot_traffic, primary_sound_type_urban, other_sound_type_urban, primary_sound_type_nature, other_sound_type_nature, insects_count, birds_count, trees_presence, native_or_wild_plant_presence, ornnamental_or_planted_plant_presence, moss_or_lichen_presence, wellbeing_rating, ease_rating, stress_rating, nature_connection_experience_rating, most_notable_nature_element, other_notes

# Cleaning the sound column
- Define a dictionary that maps each ordinal category to a numeric value where Quiet becomes 0, Moderate becomes 1, and Loud becomes 2 (preserving the order of intensity).

In [17]:
sound_mapping = {
    'Quiet': 0,
    'Moderate': 1,
    'Loud': 2
}

df['Sound_level_encoded'] = df['sound level'].map(sound_mapping)
print("Mapping Verification")
print(df[['sound level', 'Sound_level_encoded']].drop_duplicates().reset_index(drop=True))

Mapping Verification
  sound level  Sound_level_encoded
0       Quiet                    0
1    Moderate                    1
2        Loud                    2


In [18]:
# Map null/NaN values to 0 while keeping the original note text intact
df['Other_notes_encoded'] = df['other_notes'].fillna(0)

print("Mapping Verification")
print(df[['other_notes', 'Other_notes_encoded']].drop_duplicates().reset_index(drop=True))

Mapping Verification
                                          other_notes  \
0                                                 NaN   
1                                    Football in Bush   
2                                           Generator   
3                                               Trash   
4                                           Ambulance   
5                             A soft yellow golf ball   
6                                      People smoking   
7                      Bags left outside library wall   
8                                  Loud traffic noise   
9                              Lots of dumped clothes   
10                                    Yellow Golfball   
11                                            Smoking   
12                                            pathway   
13                                           gunshots   
14                                  Construction site   
15                                             People   
16        

In [19]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

display(df)

,recorder_id,date,location,weather,temperature_c,sound level,foot_traffic,primary_sound_type_urban,other_sound_type_urban,primary_sound_type_nature,other_sound_type_nature,insects_count,birds_count,trees_presence,native_or_wild_plant_presence,ornnamental_or_planted_plant_presence,moss_or_lichen_presence,wellbeing_rating,ease_rating,stress_rating,nature_connection_experience_rating,most_notable_nature_element,other_notes,Weather_mapping_encoded,Sound_level_encoded,Other_notes_encoded
0,R26_001,2026-09-22,L01 MainEntrance,Partly Cloudy,16°C,Quiet,Low,Human voices,Traffic,Wind,Trees,0,1,Many,Some,Some,Some,Positive wellbeing,Neutral,Stressful,Less connected,Plants and trees,NaN,1,0,0
1,R26_001,2026-09-22,L02 LibraryGreen,Sunny,16°C,Moderate,Low,Traffic,Human voices,Wind,Trees,0,2,Many,NaN,Some,NaN,Positive wellbeing,Neutral,Neutral,A little more connected,Sunrays on grass,NaN,0,1,0
2,R26_001,2026-09-22,L06 HedgingBehindSportsCentre,Partly Cloudy,16°C,Quiet,Low,Construction,Traffic,Wind,Other birds,10,20,Many,NaN,Many,Some,Positive wellbeing,Easy,Mostly calm,No change,Very quiet,NaN,1,0,0
3,R26_001,2026-09-22,L08 AccessRoadNearSportsCentre,Partly Cloudy,16°C,Moderate,Low,Human voices,Human voices,Wind,Other birds,0,2,One or two,Some,NaN,Some,Positive wellbeing,Neutral,Neutral,Much more connected,Harsh wind,NaN,1,1,0
4,R26_001,2026-09-22,L06 HedgingBehindSportsCentre,Sunny,16°C,Moderate,Low,Traffic,Other,Wind,Trees,0,0,One or two,NaN,Some,NaN,Positive wellbeing,Easy,Mostly calm,Less connected,Crowded area,NaN,0,1,0
5,R26_002,2026-09-22,L01 MainEntrance,Sunny,17°C,Moderate,Low,Traffic,Human voices,Other birds,Insects,2,4,Many,Some,Some,Many,Positive wellbeing,Easy,Mostly calm,A little more connected,trees,Football in Bush,0,1,Football in Bush
6,R26_002,2026-09-22,L04 PathFrontiHub,Windy,17°C,Moderate,Low,Traffic,Other,Wind,Insects,2,2,Many,Some,Many,Some,Neutral,Neutral,Very calm,No change,plants and trees,NaN,3,1,0
7,R26_002,2026-09-22,L05 GreenFrontSportsCentre,Sunny,19°C,Moderate,Low,Traffic,Service machinery,Wind,Other,7,0,Many,Many,Many,Many,Positive wellbeing,Vey easy,Mostly calm,Much more connected,wild plants,NaN,0,1,0
8,R26_002,2026-09-22,L06 HedgingBehindSportsCentre,Cloudy,16°C,Quiet,Low,Traffic,Human voices,Wind,Trees,3,1,Many,Many,Many,Some,Neutral,Neutral,Very calm,Less connected,shaded area,NaN,2,0,0
9,R26_002,2026-09-22,L07 CarParkNearSportsCentre,Sunny,18°C,Quiet,Low,Service machinery,Traffic,Other birds,Trees,6,2,Many,Many,Many,Some,Very positive wellbeing,Vey easy,Very calm,Much more connected,insects,NaN,0,0,0
